In [2]:
# ============================================
# CWT Crypto Trading Bot - FIXED VERSION
# For CrowdWisdomTrading Internship
# ============================================

print("=" * 60)
print("CWT CRYPTO TRADING BOT")
print("=" * 60)

# Step 1: Install required libraries
print("\n📦 Installing libraries...")
!pip install yfinance pandas numpy ta > /dev/null 2>&1
print("✅ Libraries installed!")

# Step 2: Import libraries
import yfinance as yf
import pandas as pd
import numpy as np
import time
import json
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("\n📊 Libraries loaded successfully!")

# ============================================
# TASK 2: Fetch last 1000 bars for crypto asset
# ============================================

def fetch_crypto_bars(symbol, interval='5m', limit=200):
    """
    Fetch historical price data for crypto
    Using yfinance (works in Colab, no API key needed)
    """
    # Map crypto symbols to yfinance format
    symbol_map = {
        'bitcoin': 'BTC-USD',
        'ethereum': 'ETH-USD',
        'solana': 'SOL-USD'
    }

    ticker = symbol_map.get(symbol.lower(), 'BTC-USD')

    # For 5m bars, 200 bars = ~16 hours
    period = '2d'

    print(f"  Fetching {limit} {interval} bars for {symbol}...")

    # Download data
    data = yf.download(ticker, period=period, interval=interval, progress=False)

    # Take last 'limit' bars
    bars = data.tail(limit)

    print(f"  ✅ Fetched {len(bars)} bars")
    return bars

# ============================================
# TASK 1: Get predictions (simplified using price data)
# ============================================

def get_price_predictions(bars):
    """
    Predict next 5min move based on recent price action
    """
    if len(bars) < 20:
        return {'up': 0.5, 'down': 0.5, 'current_price': 0, 'rsi': 50}

    # Get close prices as simple list (FIXED)
    closes = bars['Close'].values.flatten().tolist()
    current_price = closes[-1]

    # Simple momentum indicator
    price_change_5min = (closes[-1] - closes[-2]) / closes[-2] if len(closes) > 1 else 0
    price_change_30min = (closes[-1] - closes[-6]) / closes[-6] if len(closes) > 6 else 0

    # Simple RSI calculation (FIXED - no numpy mean issues)
    gains = []
    losses = []
    for i in range(1, len(closes)):
        change = closes[i] - closes[i-1]
        if change > 0:
            gains.append(change)
            losses.append(0)
        else:
            gains.append(0)
            losses.append(abs(change))

    # Calculate averages manually
    if len(gains) >= 14:
        avg_gain = sum(gains[-14:]) / 14
        avg_loss = sum(losses[-14:]) / 14
    elif len(gains) > 0:
        avg_gain = sum(gains) / len(gains)
        avg_loss = sum(losses) / len(losses)
    else:
        avg_gain = 0
        avg_loss = 1

    if avg_loss == 0:
        rsi = 70
    else:
        rs = avg_gain / avg_loss
        rsi = 100 - (100 / (1 + rs))

    # Combine signals
    up_score = 0.5
    down_score = 0.5

    if price_change_5min > 0.002:  # 0.2% up in last 5min
        up_score += 0.1
    elif price_change_5min < -0.002:
        down_score += 0.1

    if price_change_30min > 0.005:
        up_score += 0.1
    elif price_change_30min < -0.005:
        down_score += 0.1

    if rsi < 30:
        up_score += 0.15
    elif rsi > 70:
        down_score += 0.15

    # Normalize
    total = up_score + down_score
    up_prob = up_score / total
    down_prob = down_score / total

    return {
        'up': round(up_prob, 3),
        'down': round(down_prob, 3),
        'current_price': round(float(current_price), 2),
        'rsi': round(rsi, 1)
    }

# ============================================
# TASK 4: Risk Management using Kelly Formula
# ============================================

def calculate_kelly(win_probability, odds=2.0):
    """
    Kelly Formula: f* = (p * b - q) / b
    """
    if win_probability <= 0.5:
        return 0

    b = odds - 1
    p = win_probability
    q = 1 - p

    if b <= 0:
        return 0

    kelly = (p * b - q) / b

    # Cap at 25% of portfolio
    return max(0, min(kelly, 0.25))

# ============================================
# TASK 5: Feedback Loop
# ============================================

class FeedbackLoop:
    def __init__(self):
        self.prediction_history = []
        self.correct_predictions = 0
        self.total_predictions = 0

    def record_prediction(self, asset, predicted_move, predicted_prob, actual_move):
        was_correct = (predicted_move == actual_move)

        self.prediction_history.append({
            'timestamp': str(datetime.now()),
            'asset': asset,
            'predicted': predicted_move,
            'predicted_prob': predicted_prob,
            'actual': actual_move,
            'correct': was_correct
        })

        if was_correct:
            self.correct_predictions += 1
        self.total_predictions += 1

        return was_correct

    def get_accuracy(self):
        if self.total_predictions == 0:
            return 0
        return self.correct_predictions / self.total_predictions

    def get_feedback_message(self):
        accuracy = self.get_accuracy()

        if accuracy > 0.6:
            return "Strategy is working well! Continue with current approach."
        elif accuracy > 0.55:
            return "Strategy is slightly better than random. Consider improving features."
        elif accuracy > 0.5:
            return "Strategy is barely better than coin flip. Need improvement."
        else:
            return "Strategy is not working. Need significant changes."

# ============================================
# TASK 6: Scale - Multiple Assets
# ============================================

class CryptoBot:
    def __init__(self):
        self.assets = ['bitcoin', 'ethereum']
        self.feedback = FeedbackLoop()
        self.portfolio_size = 10000

    def analyze_asset(self, asset):
        print(f"\n{'='*50}")
        print(f"ANALYZING: {asset.upper()}")
        print(f"{'='*50}")

        # Fetch data
        bars = fetch_crypto_bars(asset, interval='5m', limit=200)

        if len(bars) < 20:
            print(f"  ⚠️ Not enough data for {asset}")
            return None

        # Get prediction
        prediction = get_price_predictions(bars)

        print(f"\n  📊 Current Price: ${prediction['current_price']}")
        print(f"  📈 RSI (14): {prediction['rsi']}")
        print(f"\n  🎯 PREDICTION for next 5min:")
        print(f"     Up:   {prediction['up']*100:.1f}%")
        print(f"     Down: {prediction['down']*100:.1f}%")

        # Determine predicted direction
        if prediction['up'] > prediction['down']:
            predicted_move = 'up'
            win_prob = prediction['up']
            print(f"\n  🔮 Prediction: Price will go UP")
        else:
            predicted_move = 'down'
            win_prob = prediction['down']
            print(f"\n  🔮 Prediction: Price will go DOWN")

        # Calculate position size using Kelly
        if win_prob > 0.55:
            kelly = calculate_kelly(win_prob, odds=2.0)
            position_size = self.portfolio_size * kelly
            print(f"\n  💰 KELLY POSITION SIZING:")
            print(f"     Win probability: {win_prob*100:.1f}%")
            print(f"     Kelly fraction: {kelly*100:.1f}%")
            print(f"     Suggested bet: ${position_size:.2f}")
        else:
            print(f"\n  ⏸️ No trade recommended (win prob too low)")
            position_size = 0

        return {
            'asset': asset,
            'predicted_move': predicted_move,
            'win_probability': win_prob,
            'position_size': position_size,
            'current_price': prediction['current_price']
        }

    def run_simulation(self):
        print("\n" + "=" * 60)
        print("RUNNING TRADING SIMULATION")
        print("=" * 60)

        results = []

        for asset in self.assets:
            result = self.analyze_asset(asset)
            if result:
                results.append(result)

        # Print summary
        print("\n" + "=" * 60)
        print("SUMMARY")
        print("=" * 60)

        total_bet = 0
        for r in results:
            print(f"\n  {r['asset'].upper()}:")
            print(f"    Prediction: {r['predicted_move']}")
            print(f"    Confidence: {r['win_probability']*100:.1f}%")
            print(f"    Suggested bet: ${r['position_size']:.2f}")
            total_bet += r['position_size']

        print(f"\n  💵 Total suggested exposure: ${total_bet:.2f}")
        print(f"  📊 Remaining portfolio: ${self.portfolio_size - total_bet:.2f}")

        accuracy = self.feedback.get_accuracy()
        print(f"\n  🎯 Current prediction accuracy: {accuracy*100:.1f}%")
        print(f"  📝 Feedback: {self.feedback.get_feedback_message()}")

        return results

# ============================================
# RUN THE BOT
# ============================================

print("\n" + "=" * 60)
print("STARTING CWT CRYPTO TRADING BOT")
print("=" * 60)

# Create and run bot
bot = CryptoBot()
results = bot.run_simulation()

print("\n" + "=" * 60)
print("BOT EXECUTION COMPLETE")
print("=" * 60)

CWT CRYPTO TRADING BOT

📦 Installing libraries...
✅ Libraries installed!

📊 Libraries loaded successfully!

STARTING CWT CRYPTO TRADING BOT

RUNNING TRADING SIMULATION

ANALYZING: BITCOIN
  Fetching 200 5m bars for bitcoin...
  ✅ Fetched 200 bars

  📊 Current Price: $76477.5
  📈 RSI (14): 37.1

  🎯 PREDICTION for next 5min:
     Up:   50.0%
     Down: 50.0%

  🔮 Prediction: Price will go DOWN

  ⏸️ No trade recommended (win prob too low)

ANALYZING: ETHEREUM
  Fetching 200 5m bars for ethereum...
  ✅ Fetched 200 bars

  📊 Current Price: $2319.0
  📈 RSI (14): 30.7

  🎯 PREDICTION for next 5min:
     Up:   50.0%
     Down: 50.0%

  🔮 Prediction: Price will go DOWN

  ⏸️ No trade recommended (win prob too low)

SUMMARY

  BITCOIN:
    Prediction: down
    Confidence: 50.0%
    Suggested bet: $0.00

  ETHEREUM:
    Prediction: down
    Confidence: 50.0%
    Suggested bet: $0.00

  💵 Total suggested exposure: $0.00
  📊 Remaining portfolio: $10000.00

  🎯 Current prediction accuracy: 0.0%
  